In [ ]:
import tqdm
import ffmpeg
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import os
import re 
import glob


rcParams.update({
    "font.size":        16,        
    "axes.titlesize":   16,
    "axes.labelsize":   16,
    "xtick.labelsize":  16,
    "ytick.labelsize":  16,
    "legend.fontsize":  16,
    "figure.titlesize": 16,
})

os.chdir(os.path.dirname(os.getcwd()))
print(os.getcwd())

In [ ]:
EXPERIMENT_RESULTS_PATH = 'IS26-experiments/dataset-spanishad'

In [ ]:
all_metadata = []
dataset_re = r"/dataset-([^/]+)/"
subset_re = r"/subset-([^/]+)/"
for metadata_path in glob.glob(f'{EXPERIMENT_RESULTS_PATH}/**/metadata.pkl', recursive=True):
    df = pd.read_pickle(metadata_path)
    df['dataset'] = re.search(dataset_re, metadata_path).group(1)
    df['subset'] = re.search(subset_re, metadata_path).group(1)
    df['resampled'] = 'resampled' in str(metadata_path)
    df = df.drop_duplicates('sample_id')
    all_metadata.append(df)

all_metadata = pd.concat(all_metadata, ignore_index=True)
all_metadata['experiment_description'] = all_metadata.apply(lambda row: row['dataset'] + '\n(' + row['subset'] + ' - 16KHz)' if row['resampled'] else row['dataset'] + '\n(' + row['subset'] + ')', axis=1)

print(f'Amount of experiments: {len(all_metadata.experiment_description.unique())}')

In [ ]:
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

summarize_experiments = all_metadata.groupby(['experiment_description', 'condition']).size().reset_index(name='count')
ax = sns.barplot(data=summarize_experiments, y='experiment_description', x='count', hue='condition', palette='viridis')

plt.title('Sample Distribution by Condition across Datasets and Subsets', fontsize=16, pad=20)
plt.xlabel('Sample Count (Number of sample_id)', fontsize=12)
plt.ylabel('Dataset', fontsize=12)
plt.legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
def flatten_data(data):
    reformat_data = {}
    for k, v in data.items():
        if isinstance(v, dict):
            v = {f'{k}_{kv}': vv for kv, vv in v.items()}
            reformat_data.update(flatten_data(v))
        elif isinstance(v, list):
            if len(v) != 1:
                v = {f'{k}_{i}_{kv}': vv for i, v_elem in enumerate(v) for kv, vv in v_elem.items()}
            else:
                v = {f'{k}_{kv}': vv for kv, vv in v[0].items()}
            reformat_data.update(flatten_data(v))
        else:
            reformat_data[k] = v
    return reformat_data

In [ ]:
files_to_analyze = all_metadata.file.unique()
print(f'Amount of unique audio files: {len(files_to_analyze)}')

audio_metadata = []
for audio_file in tqdm.tqdm(files_to_analyze):
    if Path(audio_file).exists():
        df = flatten_data(ffmpeg.probe(audio_file))
        audio_metadata.append(df)
    else:
        print(f'File not found: {audio_file}')

audio_metadata = pd.DataFrame(audio_metadata)
audio_metadata_columns = list(set(audio_metadata.columns) - set(['format_filename', 'format_tags_title', 
                                                                 'streams_duration', 'format_duration', 
                                                                 'streams_duration_ts', 'format_bit_rate',
                                                                 'format_size']))
all_metadata = all_metadata.merge(audio_metadata, left_on='file', right_on='format_filename', how='left')

In [ ]:
columns = ['experiment_description', 'condition'] + ['streams_codec_name', 'streams_sample_rate', 'streams_channels', 'streams_bits_per_sample']
all_metadata.groupby(columns).size().reset_index(name='counts')

In [ ]:
all_experiments_results = []
repetitions = 100
for (exp_name, dataset, subset), exp_audio_metadata in all_metadata.groupby(['experiment_description', 'dataset', 'subset']):
    print(f'Running experiment over samples in {dataset} dataset ({subset})')
    y = exp_audio_metadata['condition'].to_numpy()

    for col in audio_metadata_columns:
        if exp_audio_metadata[col].dtype != int:
            le = LabelEncoder()
            exp_audio_metadata[col] = le.fit_transform(exp_audio_metadata[col])

    X = exp_audio_metadata[audio_metadata_columns].to_numpy()

    results = []
    for i in tqdm.tqdm(range(repetitions)):
        train_features, test_features, train_labels, test_labels = train_test_split(X, y, test_size=0.2, random_state=i, stratify=y, shuffle=True)
        clf = RandomForestClassifier(n_estimators=10, random_state=i)
        clf = clf.fit(train_features, train_labels)
        test_predictions = clf.predict_proba(test_features)[:, 1]

        results.append({
            'experiment_name': exp_name,
            'subset': subset,
            'dataset': dataset,
            'repetition': i,
            'Accuracy': accuracy_score(test_labels, test_predictions > 0.5),
            'AUC': roc_auc_score(test_labels, test_predictions)
        })

    all_experiments_results.extend(results)
all_experiments_results = pd.DataFrame(all_experiments_results)


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 10), squeeze=False, sharey=True, sharex=True)
for ax, metric in zip(axes.flatten(), ['Accuracy', 'AUC']):
    sns.boxplot(data=all_experiments_results, x="experiment_name", y=metric, ax=ax, legend=False)
    sns.stripplot(data=all_experiments_results, x="experiment_name", y=metric, color=".2", alpha=0.5, jitter=True, ax=ax)
    ax.set_ylabel(metric)
    ax.set_xlabel("Subset")
    if metric == 'AUC':
        ax.axhline(0.5, color='indianred', linestyle='--', label='Random Chance')
fig.suptitle(f"Experiment Results for {exp_name.upper()}", fontsize=20, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.show()
